# Lab 03B: A CrewAI Research Crew — Context Handoff + Memory

**Week 3 — Agentic AI: Building Autonomous Intelligent Systems**

In Lab 03A you built reflection and memory *by hand*. Now you'll use a real framework, **CrewAI**, where **agents**, **tasks**, and **memory** are first-class building blocks. You'll build a three-agent research crew — a **researcher**, an **analyst**, and a **writer** — where each agent's output becomes the next agent's **context**. Then you'll turn on CrewAI's built-in **memory** and see how it lets the crew carry context *across runs*, not just within one.

## Introduction

This lesson answers:

- What is a CrewAI crew (Agents, Tasks, Crew, Process)?
- How does one agent hand its work off to the next?
- What is the difference between **task-local state** (context within a run) and **remembered context** (memory across runs)?
- What does a framework give you that hand-writing the loop does not?

## Learning Goals

After completing this lesson you will be able to:

- Define **Agents** (role / goal / backstory) and **Tasks** (description / expected_output).
- Chain tasks so each one's output becomes the next task's **`context`**.
- Run a sequential **Crew** and read each task's output.
- Enable CrewAI **memory** and explain task-local state vs. remembered context across runs.

This lab is on the **API-key** track and talks to Gemini through CrewAI + LiteLLM.

## What is a CrewAI crew?

CrewAI models a workflow as a small team. You define **Agents** (each a persona with a `role`, `goal`, and `backstory`), give them **Tasks** (each a `description` + `expected_output`), and assemble them into a **Crew** that runs the tasks in order (`Process.sequential`). The magic is **`context`**: you list which earlier tasks a task depends on, and CrewAI feeds those outputs in automatically — that is the agent handoff.

```
   topic
     │
     ▼
   ┌──────────────┐  research brief
   │  researcher  │──────────────┐
   └──────────────┘              │  context
                                 ▼
                          ┌──────────────┐  analysis
                          │   analyst    │──────────────┐
                          └──────────────┘              │  context
                                                        ▼
                                                 ┌──────────────┐  final report
                                                 │    writer    │────────────▶
                                                 └──────────────┘
   each task's output becomes the next task's `context`.
   turn on Crew memory and that context can also persist ACROSS runs.
```

## Use cases

A sequential crew fits any task with clear, ordered phases:

- **Research -> analysis -> writing** (this lab): gather, interpret, present.
- **Draft -> review -> revise**: a writer, a critic, an editor.
- **Plan -> build -> test**: a planner, a coder, a tester.
- **Extract -> normalize -> report**: a data pipeline of specialist steps.

If steps are *independent* rather than ordered, you would fan them out in parallel instead (Week 2); a crew is for handoffs.

## Building blocks

- **Agent** — a worker defined by `role`, `goal`, and `backstory` (always a senior/expert persona here).
- **Task** — a `description` (what to do), an `expected_output` (the shape), and the `agent` that owns it.
- **`context`** — the list of earlier tasks whose outputs feed this one (the handoff).
- **Crew + Process** — the agents + tasks + an execution order (`sequential`).
- **Memory (optional)** — CrewAI's built-in short-term / long-term / entity memory, which carries context across runs.
- **An embedder** — memory stores and retrieves by similarity, so it needs an embedding model (we point it at Gemini).

## Considerations for trustworthy crews

- **Bound the crew.** More agents and hops mean more cost and more places to go wrong; add only the steps the task needs.
- **Context is not free.** Every handoff injects the prior output into the next prompt — long chains grow the context fast.
- **Memory is powerful and sticky.** Remembered context helps continuity but can also carry stale or wrong facts forward; know what is being persisted.
- **Constrain outputs where it matters.** Use `expected_output` (and schemas) so a downstream agent receives a predictable shape.
- **Keep handoffs inspectable.** Read each `task.output` so you can see exactly what each agent contributed.

## Setup: add your Gemini API key as a Colab secret

1. Get a key from [Google AI Studio](https://aistudio.google.com/app/apikey).
2. In Colab, click the **key icon** in the left sidebar ("Secrets").
3. Add a new secret named **`GEMINI_API_KEY`** and paste your key as the value.
4. Toggle **"Notebook access"** on for that secret.

The next cell installs CrewAI (which pulls in LiteLLM, the layer that lets CrewAI call Gemini). This is a larger install — give it a minute.

In [ ]:
!pip install -q crewai

> **Heads-up on the pip output:** you may see an `ERROR: pip's dependency resolver ...` line mentioning `bigframes` and `rich`. This is **expected in Colab and safe to ignore** — CrewAI upgrades `rich`, and Colab's preinstalled `bigframes` pins an older `rich`; this lab never uses `bigframes`, and the install still succeeded. If Colab shows a **"Restart session"** prompt after installing, click it (or **Runtime -> Restart session**) and re-run from the setup cell.

In [ ]:
import os

from google.colab import userdata
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

load_dotenv()

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("Missing GEMINI_API_KEY. Add it in Colab Secrets (key icon) and enable notebook access.")

# LiteLLM (under CrewAI) reads the key from this env var for gemini/* models.
os.environ["GEMINI_API_KEY"] = api_key

# The `gemini/` prefix routes to the Gemini API. NOTE: this lab uses gemini-2.5-flash, not the
# gemini-2.5-flash-lite the other labs use. A crew fires many larger, multi-step requests, and the
# lite model -- the cheapest and most heavily used -- is often overloaded (HTTP 503) for requests
# that size, while flash has the capacity to serve them reliably. num_retries adds a per-call retry
# as an extra cushion for the occasional transient blip.
llm = LLM(
    model="gemini/gemini-3.5-flash",
    api_key=api_key,
    temperature=0.3,
    num_retries=5,
)

# While CrewAI/LiteLLM is quietly retrying a 503, it still logs alarming red "ERROR" lines.
# Quiet those so the notebook output stays readable; our own retry messages (plain prints) still show.
import logging
for _noisy in ("crewai.flow.runtime", "LiteLLM", "litellm", "root"):
    logging.getLogger(_noisy).setLevel(logging.CRITICAL)

# ChromaDB's Google embedder (used by CrewAI memory) still imports the legacy google.generativeai
# package and prints a deprecation notice. It works fine; silence the notice to keep output clean.
import warnings
warnings.filterwarnings("ignore", message=r".*google\.generativeai.*")

# CrewAI can upload run "traces" to its hosted dashboard. We keep it off so nothing leaves this
# notebook and CrewAI stops printing its "Tracing Preference Saved" panel. (See the note by the run
# cell -- the dashboard is genuinely useful for your own later projects; flip this to "true" to try it.)
os.environ["CREWAI_TRACING_ENABLED"] = "false"

# Quick connectivity check. Gemini can return a transient 503 ("high demand") on any call,
# so we retry a few times, and if it still will not respond we warn instead of crashing.
import time

def _health_check(prompt, tries=4, wait=8):
    for attempt in range(1, tries + 1):
        try:
            return llm.call(prompt)
        except Exception as e:
            if "503" in str(e) and attempt < tries:
                print(f"  Model overloaded (503). Waiting {wait}s, then retrying ({attempt}/{tries})...")
                time.sleep(wait)
            else:
                raise

try:
    print(_health_check("Say 'Setup complete!' and nothing else."))
except Exception as e:
    print("Connectivity check could not complete (model may be busy):", str(e)[:120])
    print("That is OK -- num_retries retries a transient 503 when you run the crew below.")

## The three agents

Each agent is a senior persona. They share the same model; what makes one a researcher and another a writer is the `role` / `goal` / `backstory`.

In [ ]:
TOPIC = "the impact of AI agents on software development workflows"


researcher = Agent(
    role="Senior Research Analyst",
    goal="Gather a faithful, specific research brief on the topic.",
    backstory="You dig up the key facts, real challenges, and concrete examples; you cite concepts, not hype.",
    llm=llm,
    verbose=False,
)

analyst = Agent(
    role="Senior Strategy Analyst",
    goal="Turn research into ranked implications and clear recommendations.",
    backstory="You are opinionated and evidence-driven; you separate what matters from what does not.",
    llm=llm,
    verbose=False,
)

writer = Agent(
    role="Senior Technical Writer",
    goal="Turn analysis into a polished, readable short report.",
    backstory="You write tight, engaging prose for a general audience; no jargon dumps.",
    llm=llm,
    verbose=False,
)

## The three tasks -- and the handoff via `context`

Each task's `description` follows the Role / Context / Task / Constraints / Format shape. The part that makes this a *crew* and not three separate calls is the **`context=[...]`** argument on each `Task` -- that list **is** the handoff.

When you write `context=[research_task]` on the analyst's task, CrewAI takes whatever `research_task` produced and **injects it into the analyst's prompt automatically**. You never copy the brief across by hand; you just name the upstream task. The writer names both earlier tasks, so it receives both outputs.

Reading the `context=` lines top to bottom is the wiring diagram of the crew:

```
research_task    context=[]                          -> runs first; no upstream input
                       |
                       |  its output is injected as context
                       v
analysis_task    context=[research_task]             -> sees the researcher's brief
                       |
                       |  both outputs injected
                       v
writing_task     context=[research_task,             -> sees BOTH the brief
                          analysis_task]                 and the analysis
```

So the handoff is declarative: instead of passing arguments between function calls, each task *declares which earlier tasks it depends on*, and CrewAI threads the outputs through. The topic is embedded directly in the first task, so we call `kickoff()` with no templated inputs.

In [ ]:
research_task = Task(
    description=f"""# Context
You are the first step; the analyst and writer build on your brief.

# Task
Produce a concise research brief on the topic in the Input section.

# Constraints
- Cover key facts, main challenges, and 2-3 notable examples.
- Be specific; cite concepts, not URLs.

# Format
Short sections with clear headers.

# Input (topic)
{TOPIC}""",
    expected_output="A structured research brief with clear section headers.",
    agent=researcher,
)

analysis_task = Task(
    description="""# Context
The researcher's brief is available to you as context.

# Task
Analyze the research and draw the most important implications and recommendations.

# Constraints
- Rank implications by impact; back each claim with reasoning.

# Format
Executive Summary (2 sentences), Key Implications (3 bullets), Recommendations (2 bullets).""",
    expected_output="A ranked analysis with a summary, implications, and recommendations.",
    agent=analyst,
    context=[research_task],
)

writing_task = Task(
    description="""# Context
The research brief AND the analysis are available to you as context.

# Task
Write a polished, publication-ready short report that combines them for a general audience.

# Constraints
- 200-300 words; clear and engaging; no jargon dumps.

# Format
A title, then 2-3 short paragraphs.""",
    expected_output="A short titled report of 200-300 words.",
    agent=writer,
    context=[research_task, analysis_task],
)

## A note on 503 "model overloaded" errors

Gemini can return a transient **503 ("high demand")** when a model is busy. Two choices keep this lab robust:

1. **Model choice:** this lab uses `gemini-2.5-flash` (see setup). A crew fires many larger requests, and the cheaper `gemini-2.5-flash-lite` is frequently overloaded for that load, while `flash` has the capacity to serve it.
2. **Per-call retry:** `num_retries=5` on the `LLM` retries an individual call on a transient blip before it can fail the crew.

If Gemini has a broad capacity event, calls can still fail -- that is a server-side outage, not a bug. Wait a few minutes and run the cell again.

## Run the crew

`Process.sequential` runs the tasks in order and threads the `context` through. After the run, each task's result is on `task.output` — read them to see exactly what each agent handed off.

> **Async note (Colab):** Colab already runs an `asyncio` event loop, and this version of CrewAI refuses a *synchronous* `crew.kickoff()` from inside a running loop. So we use the async entry point **`await crew.kickoff_async()`** with top-level `await` — exactly the pattern from the async lab. (In a plain `.py` script with no running loop, `crew.kickoff()` works directly.)

> **Aside — CrewAI's tracing dashboard:** CrewAI can upload a step-by-step trace of each run (every agent, the exact prompts and responses, token counts, latency) to a hosted web dashboard for debugging. We keep it **off** in this lab (`CREWAI_TRACING_ENABLED=false` in setup) so nothing leaves your notebook and the output stays clean — everything you need is printed inline below. For your own real projects it is worth a look: set `tracing=True` on the `Crew` (it needs a free CrewAI account) to inspect runs in the UI.

In [ ]:
crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[research_task, analysis_task, writing_task],
    process=Process.sequential,
    verbose=False,
)

try:
    await crew.kickoff_async()

    print("=== RESEARCH BRIEF (researcher) ===")
    print(research_task.output.raw)
    print("\n=== ANALYSIS (analyst -- read the brief via context) ===")
    print(analysis_task.output.raw)
    print("\n=== FINAL REPORT (writer -- read both via context) ===")
    print(writing_task.output.raw)
except Exception as e:
    if "503" in str(e):
        print("Gemini is under heavy load right now (repeated 503s) and the crew could not finish.")
        print("This is a transient, server-side capacity issue -- not a bug in your code or this lab.")
        print("Wait a few minutes and re-run this cell; single calls usually recover quickly.")
    else:
        raise

## What just happened: context handoff

You never passed the research text to the analyst by hand. Because `analysis_task` declared `context=[research_task]`, CrewAI injected the researcher's output into the analyst's prompt automatically; the writer got both. That is the crew handoff — coordination through `context` instead of manual argument passing.

But notice: that context is **task-local**. It lives only for this one `kickoff()`. Run the crew again and it starts fresh with no memory of the last run. Enabling **memory** changes that.

## Optional: enable CrewAI memory (context that survives across runs)

Set `memory=True` on the Crew and CrewAI keeps short-term, long-term, and entity memory *across* `kickoff()` calls — so a later run can recall what earlier runs established. Because memory retrieves by similarity, it needs an **embedder**; we point it at Gemini's embedding model so the lab stays Gemini-only (the default embedder is OpenAI).

The contrast to feel:
- **Without memory** (above): every run is isolated — only task-local context within the run.
- **With memory** (below): the crew accumulates a memory store that later runs read from.

> **Heads-up:** memory + the embedder config can vary by CrewAI version and needs a live Colab run to confirm. If the embedder line errors, check the CrewAI memory docs for the provider/model names your installed version expects.

> As above, we use `await memory_crew.kickoff_async()` because Colab has a running event loop.

In [ ]:
memory_crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[research_task, analysis_task, writing_task],
    process=Process.sequential,
    memory=True,  # <-- turn on short-term / long-term / entity memory across runs
    embedder={
        # CrewAI renamed the Gemini embeddings provider: use "google-generativeai" (the Gemini API
        # path) -- "google-vertex" is the separate Vertex AI path, and the old bare "google" is gone.
        "provider": "google-generativeai",
        "config": {"api_key": api_key, "model_name": "gemini-embedding-001"},
    },
    verbose=False,
)

# Run 1 seeds the memory; Run 2 can recall it. (Same crew, run twice.)
try:
    print("--- Run 1 (seeds memory) ---")
    await memory_crew.kickoff_async()
    print(writing_task.output.raw[:300], "...")

    print("\n--- Run 2 (can recall Run 1 from memory) ---")
    await memory_crew.kickoff_async()
    print(writing_task.output.raw[:300], "...")
except Exception as e:
    if "503" in str(e):
        print("Gemini is under heavy load right now (repeated 503s); the memory demo could not finish.")
        print("This is a transient server-side issue -- wait a few minutes and re-run this cell.")
    else:
        raise

## Your turn (exercises)

1. **Add a fourth agent.** Insert a "Senior Fact-Checker" task between the researcher and the analyst that flags any claim the brief can't support; feed it forward with `context`.
2. **Structured handoff.** Give `analysis_task` an `output_pydantic` model so the writer receives typed, predictable analysis instead of prose.
3. **Swap the process.** Try `Process.hierarchical` (with a manager LLM) and observe how task delegation changes.
4. **Prove memory works.** With `memory=True`, run the crew on a topic, then run it again asking it to "build on what you found last time" and check whether Run 2 references Run 1.
5. **Compare to Lab 03A.** You built memory by hand there and got it from the framework here. Which was clearer? Which would you reach for in production, and why?

When you're done, save a copy (**File -> Save a copy in Drive**) and submit your notebook link via Canvas.

---
---

# ⬛⬛⬛  END OF THE ORIGINAL LAB  ⬛⬛⬛
# ⬛⬛⬛  MY EXERCISE WORK STARTS BELOW  ⬛⬛⬛

**Everything ABOVE this cell is the original lab, completely unmodified** — all 20 cells are
byte-for-byte identical to the version handed out. I added nothing to them, edited nothing in
them, and deleted nothing from them.

**Everything BELOW this cell is mine.** Every single added cell is tagged so you never have to
guess which is which:

| | How it is tagged |
|---|---|
| **Markdown cells** | the heading carries a `` `[ADDED · EX-n]` `` label |
| **Code cells** | the first lines are an `ADDED · EX-n` banner comment |

### Where to find each exercise

| Tag | Exercise | Cells |
|---|---|---|
| `[ADDED · SETUP]` | Shared helpers used by the exercises below | 2 |
| `[ADDED · EX-1]` | A fourth agent: the Senior Fact-Checker | 5 |
| `[ADDED · EX-2]` | Structured handoff with `output_pydantic` | 5 |
| `[ADDED · EX-3]` | `Process.hierarchical` — **deferred**, with notes | 1 |
| `[ADDED · EX-4]` | Prove memory works (with a `memory=False` control) | 4 |
| `[ADDED · EX-5]` | Compare to Lab 03A — the written answer | 1 |

### One design rule I held to throughout

No added cell ever reuses the original `research_task` / `analysis_task` / `writing_task`
objects. A `Task` stores its result on `task.output`, so handing an original task to a new
`Crew` would overwrite the baseline result in place. The baseline outputs in
`outputs/before-memory/` are the control every comparison below is measured against — so my
cells **read** them and build their own fresh `Task` objects instead.

That is why you will see a `make_research_task()` factory below whose prompt text duplicates the
original `research_task`. The duplication is deliberate: the wording is copied verbatim so the
added fact-checker is the *only* variable, and the copy exists so the original object is never
touched.

---
---

---

## `[ADDED · SETUP]` Shared helpers

Three small utilities the exercises reuse.

`make_research_task()` returns a **fresh** research task whose wording is **byte-identical to the
baseline one above**. It would be tempting to improve it — adding "always name the source of a
figure" would prevent the very drift Exercise 1 is about — but that would change two variables at
once and make the comparison worthless. The prompt stays as it was; the new *agent* is the only
thing that changes.

`run_crew()` wraps `kickoff_async()` in the same 503 guard the lab uses, so one busy moment at
Gemini's end doesn't lose a cell's work.


In [ ]:
# ==============================================================================
# ADDED · SETUP
# shared helpers used by every exercise below
# ----------------------------------------------------------------------------
# Not part of the original lab. Builds its own Task objects; never mutates the
# original research_task / analysis_task / writing_task.
# ==============================================================================

import os
import re

OUT_ROOT = "outputs"


def save_output(relpath: str, text: str) -> str:
    """Write one task output to outputs/<relpath> and return the path.

    On Colab this lands under /content/outputs -- grab it from the file browser on the left.
    Run locally, it lands in the repo alongside outputs/before-memory/.
    """
    path = os.path.join(OUT_ROOT, relpath)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as fh:
        fh.write(text)
    return path


def make_research_task(topic: str = TOPIC) -> Task:
    """A FRESH research task, worded EXACTLY like the baseline one.

    Two reasons it is a factory rather than a shared object:
      1. A Task holds its result on `.output`; reusing one instance across crews clobbers
         the earlier result, including the baseline we are comparing against.
      2. Exercises 1, 2 and 4 each need their own independent run of the same first step.

    The wording is deliberately NOT improved -- see the markdown above.
    """
    return Task(
        description=f"""# Context
You are the first step; the analyst and writer build on your brief.

# Task
Produce a concise research brief on the topic in the Input section.

# Constraints
- Cover key facts, main challenges, and 2-3 notable examples.
- Be specific; cite concepts, not URLs.

# Format
Short sections with clear headers.

# Input (topic)
{topic}""",
        expected_output="A structured research brief with clear section headers.",
        agent=researcher,
    )


async def run_crew(crew, label: str = "the crew") -> bool:
    """kickoff_async() with the lab's 503 guard. Returns True if the run completed."""
    try:
        await crew.kickoff_async()
        return True
    except Exception as e:
        if "503" in str(e):
            print(f"Gemini is under heavy load right now (repeated 503s); {label} could not finish.")
            print("That is a server-side capacity issue, not a bug. Wait a few minutes and re-run.")
            return False
        raise


print("Helpers ready.")

---

## `[ADDED · EX-1]` Exercise 1 — a fourth agent: the Senior Fact-Checker

### Why this exercise has real teeth on this notebook

The baseline run in `outputs/before-memory/` already contains the exact failure a fact-checker
should catch. One claim degrades across all three hops:

| Hop | What it says |
|---|---|
| `research_brief.md` (researcher) | "The industry standard ... is **SWE-bench** ... state-of-the-art agentic systems now resolve **over 20-30%** of these ... problems autonomously" |
| `analysis.md` (analyst) | "state-of-the-art agents can now resolve **20-30%** of complex software engineering issues autonomously" — *SWE-bench is gone* |
| `final_report.md` (writer) | "cutting-edge agents can now solve **nearly a third** of complex engineering issues" — *the figure is gone too* |

Three things went wrong, and only the first is the researcher's fault:

1. **"over 20-30%" is incoherent as written** — a floor and a range at once. An auditor should flag it.
2. **The analyst dropped the attribution.** The number survived; the benchmark that gives it meaning did not.
3. **The writer rounded an unsourced range up** into "nearly a third", which is the top of the range
   presented as the typical case.

Nobody lied. The context handoff just let a specific, sourced claim decay into a confident vague one —
and because each hop only ever sees text, not provenance, no agent in the baseline crew is in a
position to notice. That is what the new agent is for.

### The rewiring

```
ex1_research_task    context=[]
        |
        v
ex1_factcheck_task   context=[ex1_research_task]                      <-- NEW
        |
        v
ex1_analysis_task    context=[ex1_research_task, ex1_factcheck_task]
        |
        v
ex1_writing_task     context=[ex1_factcheck_task, ex1_analysis_task]
```

Note what the writer does **not** get: the raw brief. The baseline gave it
`[research_task, analysis_task]`, but the analysis already carries the brief's substance forward,
and the lab's own "context is not free" warning argues against injecting a third full document into
the last prompt. Dropping it also closes the loophole that produced the drift — the writer can no
longer reach past the audit to the unqualified original.

The fact-checker is also told, twice, **not to add facts of its own**. An auditor that starts
researching is just a second researcher, and its inventions would arrive downstream wearing the
authority of a verification step.


In [ ]:
# ==============================================================================
# ADDED · EX-1
# the fact-checker agent + the rewired four-agent crew
# ----------------------------------------------------------------------------
# Not part of the original lab. Builds its own Task objects; never mutates the
# original research_task / analysis_task / writing_task.
# ==============================================================================

fact_checker = Agent(
    role="Senior Fact-Checker",
    goal="Flag every claim in the brief that the brief itself cannot support.",
    backstory=(
        "You audit evidence for a living and you are adversarial about it. You never soften a "
        "verdict to be agreeable, and you never add facts of your own -- an auditor who starts "
        "researching is just a second researcher, and their inventions arrive downstream wearing "
        "the authority of a verification step."
    ),
    llm=llm,
    verbose=False,
)

ex1_research_task = make_research_task()

ex1_factcheck_task = Task(
    description="""# Context
The researcher's brief is available to you as context. It is the ONLY evidence you have.
You add no new research.

# Task
Audit every substantive claim in the brief -- every figure, benchmark result, named system,
and causal assertion.

# Constraints
- Give each claim exactly one verdict: SUPPORTED, UNSUPPORTED, OVERSTATED, or NEEDS-ATTRIBUTION.
- Quote the claim verbatim (trim to <= 15 words) so the agents after you can match it by string.
- Judge the claim AS WRITTEN. A number stated without the source that gives it meaning is
  NEEDS-ATTRIBUTION even if the number is plausible. A range presented as a floor
  ("over 20-30%") is OVERSTATED because it cannot be both.
- Do NOT invent replacement facts or supply the missing source. Say what is missing instead.
- Be specific about the consequence: what would a reader wrongly conclude?

# Format
A markdown table with columns: Claim | Verdict | Why.
Then a section headed exactly "## Do not repeat downstream" listing, as `- ` bullets, every
claim that must be dropped or qualified, each with the qualification it needs.""",
    expected_output=(
        "A markdown claim-audit table (Claim | Verdict | Why) followed by a "
        "'## Do not repeat downstream' bullet list."
    ),
    agent=fact_checker,
    context=[ex1_research_task],
)

ex1_analysis_task = Task(
    description="""# Context
Two documents are available to you as context: the researcher's brief, and the fact-checker's
audit of that brief.

# Task
Analyze the research and draw the most important implications and recommendations.

# Constraints
- Rank implications by impact; back each claim with reasoning.
- The audit OVERRIDES the brief. Where they disagree, the audit wins.
- Never restate a claim from the audit's "Do not repeat downstream" list without the
  qualification that list demands.
- If you cite a figure, carry its source in the same sentence.

# Format
Executive Summary (2 sentences), Key Implications (3 bullets), Recommendations (2 bullets),
then a section headed exactly "## Claims dropped" naming any claim you left out and why.""",
    expected_output=(
        "A ranked analysis with a summary, implications, recommendations, and a "
        "'## Claims dropped' section."
    ),
    agent=analyst,
    context=[ex1_research_task, ex1_factcheck_task],
)

ex1_writing_task = Task(
    description="""# Context
The fact-checker's audit AND the analysis are available to you as context. You do NOT have the
original brief -- by design. The analysis already carries its substance, and you must not be
able to reach past the audit to an unqualified original.

# Task
Write a polished, publication-ready short report for a general audience.

# Constraints
- 200-300 words; clear and engaging; no jargon dumps.
- If you use a figure, keep it exact and name its source in the same sentence.
- Never replace a specific figure with a vague quantifier ("nearly a third", "most",
  "the majority"). Either give the number with its source, or make the point without a number.
- Anything on the audit's "Do not repeat downstream" list is off limits unqualified.

# Format
A title, then 2-3 short paragraphs.""",
    expected_output="A short titled report of 200-300 words.",
    agent=writer,
    context=[ex1_factcheck_task, ex1_analysis_task],
)

ex1_crew = Crew(
    agents=[researcher, fact_checker, analyst, writer],
    tasks=[ex1_research_task, ex1_factcheck_task, ex1_analysis_task, ex1_writing_task],
    process=Process.sequential,
    verbose=False,
)

if await run_crew(ex1_crew, "the four-agent crew"):
    for header, task, fname in [
        ("RESEARCH BRIEF (researcher)", ex1_research_task, "research_brief.md"),
        ("FACT-CHECK AUDIT (fact-checker -- NEW)", ex1_factcheck_task, "factcheck_audit.md"),
        ("ANALYSIS (analyst -- brief + audit)", ex1_analysis_task, "analysis.md"),
        ("FINAL REPORT (writer -- audit + analysis)", ex1_writing_task, "final_report.md"),
    ]:
        print(f"\n{'=' * 70}\n=== {header} ===\n{'=' * 70}")
        print(task.output.raw)
        save_output(f"after-factchecker/{fname}", task.output.raw)
    print(f"\nSaved 4 files to {OUT_ROOT}/after-factchecker/")

---

### `[ADDED · EX-1]` Check: did the fact-checker actually catch the drift?
An LLM auditing an LLM is exactly the setup that produces agreeable nonsense, so the audit gets
graded by **plain Python**, not by another model. This is the deterministic-checker pattern from
Lab 03A: anything objectively decidable is decided in code, and the model is only trusted with
what genuinely needs judgment.

Three probes, none of which makes an API call:

1. **`drift_report()`** — pull every precise figure out of an upstream document and every vague
   quantifier out of a downstream one. A precise figure upstream that survives only as a vague
   quantifier downstream **is** the drift, mechanically detected.
2. **The baseline is run through the same function first**, as a positive control. If the checker
   cannot find the drift we already know is in `outputs/before-memory/`, the checker is broken and
   its clean verdict on Exercise 1 would mean nothing.
3. **The audit table is parsed** for its verdict distribution. An audit that returns
   `SUPPORTED` for everything is a rubber stamp, and a rubber stamp should fail this cell.

The baseline excerpts are inlined so the cell works on Colab, where `outputs/before-memory/`
isn't present; if the real files are on disk they are read instead.


In [ ]:
# ==============================================================================
# ADDED · EX-1
# drift detector + its positive control on the baseline
# ----------------------------------------------------------------------------
# Not part of the original lab. Builds its own Task objects; never mutates the
# original research_task / analysis_task / writing_task.
# ==============================================================================

# Deterministic drift detection -- no API calls, so this is instant and testable like any code.

# A precise figure: "20-30%", "2%", "40 percent". The range form must come first in the
# alternation, or the plain-number branch would match "20" out of "20-30%" and lose the range.
FIGURE_RE = re.compile(
    r"\b\d+(?:\.\d+)?\s*(?:-|to|--|\u2013)\s*\d+(?:\.\d+)?\s*(?:%|percent)"
    r"|\b\d+(?:\.\d+)?\s*(?:%|percent)",
    re.I,
)

# A vague quantifier standing in for a number. "a third" etc. are the substitutions that actually
# happen when a model paraphrases a range; the leading hedge is optional so bare "a third" counts.
VAGUE_RE = re.compile(
    r"\b(?:nearly|almost|about|roughly|around|some|over|under|more than|less than)?\s*"
    r"\b(?:a third|a quarter|a half|half|two[- ]thirds|three[- ]quarters|most|the majority|"
    r"the bulk|a minority|a fraction)\b",
    re.I,
)

VERDICTS = ("SUPPORTED", "UNSUPPORTED", "OVERSTATED", "NEEDS-ATTRIBUTION")


def norm_figure(s: str) -> str:
    """Normalize a figure so '20-30%' and '20 to 30 percent' compare equal."""
    s = s.lower().replace("percent", "%")
    s = re.sub(r"\s*(?:--|\u2013|\bto\b)\s*", "-", s)
    return re.sub(r"\s+", "", s)


def figures(text: str) -> set:
    return {norm_figure(m.group()) for m in FIGURE_RE.finditer(text)}


def vague(text: str) -> list:
    return [m.group().strip() for m in VAGUE_RE.finditer(text)]


def drift_report(label: str, upstream: str, downstream: str, attributions=()) -> dict:
    """Compare an upstream doc against a downstream one for three kinds of decay.

    Returns the findings dict as well as printing, so a caller can assert on it.
    """
    up_figs, down_figs = figures(upstream), figures(downstream)
    findings = {
        "figures_lost": sorted(up_figs - down_figs),
        "vague_downstream": vague(downstream),
        "attributions_lost": sorted(
            a for a in attributions if a.lower() in upstream.lower() and a.lower() not in downstream.lower()
        ),
    }
    # The signature of substitution: a precise figure vanished AND a vague quantifier appeared.
    findings["substitution"] = bool(findings["figures_lost"] and findings["vague_downstream"])

    print(f"--- {label} ---")
    print(f"  precise figures upstream:   {sorted(up_figs) or '(none)'}")
    print(f"  precise figures downstream: {sorted(down_figs) or '(none)'}")
    print(f"  figures LOST in the hop:    {findings['figures_lost'] or '(none)'}")
    print(f"  vague quantifiers added:    {findings['vague_downstream'] or '(none)'}")
    print(f"  attributions LOST:          {findings['attributions_lost'] or '(none)'}")
    print(f"  => figure-to-vague substitution: {'YES -- DRIFT' if findings['substitution'] else 'no'}")
    print()
    return findings


# --- Probe 1: positive control. The checker must find the drift we already know is there. ---
# Inlined so this runs on Colab; the real files are preferred when present.
BASELINE_FALLBACK = {
    "research_brief.md": (
        "The industry standard for evaluating these agents is SWE-bench, a dataset of real-world "
        "GitHub issues from popular open-source repositories. While early LLMs solved <2% of these "
        "issues, state-of-the-art agentic systems now resolve over 20-30% of these complex, "
        "multi-file software engineering problems autonomously."
    ),
    "analysis.md": (
        "While state-of-the-art agents can now resolve 20-30% of complex software engineering "
        "issues autonomously, their enterprise adoption is severely bottlenecked by inadequate "
        "testing infrastructure, security risks, and high operational costs."
    ),
    "final_report.md": (
        "While cutting-edge agents can now solve nearly a third of complex engineering issues, "
        "their adoption is bottlenecked by a surprising obstacle: testing."
    ),
}


def read_baseline(fname: str) -> str:
    path = os.path.join(OUT_ROOT, "before-memory", fname)
    if os.path.exists(path):
        return open(path).read()
    print(f"  (using inlined excerpt for {fname}; {path} not present)")
    return BASELINE_FALLBACK[fname]


print("=" * 70)
print("PROBE 1 -- positive control: the KNOWN drift in the baseline run")
print("=" * 70)
base_brief = read_baseline("research_brief.md")
base_analysis = read_baseline("analysis.md")
base_report = read_baseline("final_report.md")

base_hop1 = drift_report("BASELINE hop 1: brief -> analysis", base_brief, base_analysis, ["SWE-bench"])
base_hop2 = drift_report("BASELINE hop 2: analysis -> final report", base_analysis, base_report, ["SWE-bench"])

control_ok = base_hop2["substitution"] and base_hop1["attributions_lost"] == ["SWE-bench"]
print(f"positive control: {'PASS -- the checker detects real drift' if control_ok else 'FAIL -- checker is broken'}")
assert control_ok, (
    "The drift detector failed its positive control, so its verdict on Exercise 1 would be "
    "meaningless. Fix the checker before reading anything below."
)

In [ ]:
# ==============================================================================
# ADDED · EX-1
# grade the Exercise 1 run (probes 2 and 3)
# ----------------------------------------------------------------------------
# Not part of the original lab. Builds its own Task objects; never mutates the
# original research_task / analysis_task / writing_task.
# ==============================================================================

# --- Probes 2 and 3: grade the Exercise 1 run with the checker the control just validated. ---

def task_text(task):
    """The raw output of a task, or None if it never ran (e.g. the cell hit a 503)."""
    out = getattr(task, "output", None)
    return getattr(out, "raw", None) if out else None


ex1_brief = task_text(ex1_research_task)
ex1_audit = task_text(ex1_factcheck_task)
ex1_analysis = task_text(ex1_analysis_task)
ex1_report = task_text(ex1_writing_task)

if not all([ex1_brief, ex1_audit, ex1_analysis, ex1_report]):
    print("Exercise 1 has not produced all four outputs yet -- run the crew cell above first.")
else:
    print("=" * 70)
    print("PROBE 2 -- did the SAME drift survive the four-agent pipeline?")
    print("=" * 70)
    ex1_hop1 = drift_report("EX1 hop 1: brief -> analysis", ex1_brief, ex1_analysis, ["SWE-bench"])
    ex1_hop2 = drift_report("EX1 hop 2: analysis -> final report", ex1_analysis, ex1_report, ["SWE-bench"])

    print("=" * 70)
    print("PROBE 3 -- is the audit a real audit or a rubber stamp?")
    print("=" * 70)

    # Count verdicts by scanning table rows. Rows are matched loosely because a model's markdown
    # spacing varies; what matters is one verdict token per row.
    rows = [ln for ln in ex1_audit.splitlines() if ln.count("|") >= 2 and not set(ln) <= set("|-: ")]
    counts = {v: 0 for v in VERDICTS}
    for row in rows:
        # NEEDS-ATTRIBUTION contains "ATTRIBUTION" but not "SUPPORTED"; check longest first so
        # UNSUPPORTED is never miscounted as SUPPORTED.
        for verdict in sorted(VERDICTS, key=len, reverse=True):
            if verdict in row.upper():
                counts[verdict] += 1
                break

    total = sum(counts.values())
    flagged = total - counts["SUPPORTED"]
    print(f"  table rows carrying a verdict: {total}")
    for verdict in VERDICTS:
        print(f"    {verdict:<20} {counts[verdict]}")
    print(f"  claims flagged (anything but SUPPORTED): {flagged}")

    has_dnr = "do not repeat downstream" in ex1_audit.lower()
    print(f"  '## Do not repeat downstream' section present: {has_dnr}")

    # Did the auditor find the specific thing we know is wrong -- the incoherent "over 20-30%"?
    over_range = re.search(r"over\s+\d+\s*(?:-|to|--|\u2013)\s*\d+\s*%", ex1_brief, re.I)
    if over_range:
        quoted = over_range.group().lower().split()[-1]  # e.g. "20-30%"
        caught = quoted.rstrip("%") in ex1_audit or "over 20-30" in ex1_audit.lower()
        print(f"  brief again wrote an incoherent range ({over_range.group()!r}); audit mentions it: {caught}")
    else:
        print("  brief did not repeat the 'over N-M%' construction this run (nothing to catch there)")

    print()
    print("=" * 70)
    print("VERDICT")
    print("=" * 70)
    results = {
        "audit flagged at least one claim (not a rubber stamp)": flagged > 0,
        "audit produced the machine-readable handoff section": has_dnr,
        "no figure-to-vague substitution at the analyst hop": not ex1_hop1["substitution"],
        "no figure-to-vague substitution at the writer hop": not ex1_hop2["substitution"],
        "SWE-bench attribution survived to the analysis": not ex1_hop1["attributions_lost"],
    }
    for name, passed in results.items():
        print(f"  [{'PASS' if passed else 'FAIL'}] {name}")

    print()
    if all(results.values()):
        print("All checks pass: adding the auditor closed the drift the baseline shipped.")
    else:
        print("Some checks FAIL. That is a legitimate result, not a broken cell -- write down which")
        print("ones and why. A verification agent that only sometimes catches a known-bad claim is")
        print("exactly the finding Exercise 5 should be arguing about, and a 'no drift this run'")
        print("result on one sample is weak evidence either way. Re-run to see if it is stable.")

---

## `[ADDED · EX-2]` Exercise 2 — structured handoff with `output_pydantic`

The analyst currently hands the writer **prose**. Prose is a contract nobody can enforce: the
"Key Implications (3 bullets)" instruction is a request, and a downstream consumer that wants
implication #1 has to go find it with a regular expression.

`output_pydantic` replaces that with a schema. Three consequences worth naming:

1. **`analysis_task.output.pydantic` is a real object.** `.implications[0].claim` addresses a
   field. No parsing, and a missing field is an error at the boundary instead of a silent
   miscommunication three hops later.
2. **Ranking becomes checkable.** "Rank implications by impact" is unverifiable in prose. With
   `rank: int` it's an assertion — the cell below fails the run if the ranks aren't a contiguous
   1..N, which is the sort of thing prose lets slide forever.
3. **The audit gets a guaranteed lane.** This is where Exercise 2 collides with Exercise 1: the
   fact-checker's "do not repeat downstream" list is prose living in the analyst's prose output.
   Constrain the analyst to a schema with no field for dropped claims and the audit trail is the
   first thing squeezed out — the schema would silently delete the safety feature. So
   `dropped_claims` is part of the model.

Two adjustments that are easy to miss, and both break things quietly:

- **`expected_output` must describe the JSON**, not "a ranked analysis". CrewAI puts both the
  schema and `expected_output` in the prompt; a prose-shaped `expected_output` pulls against the
  schema and you get schema-shaped prose or a retry loop.
- **The writer now receives serialized JSON** in its context, not paragraphs. Its `description`
  has to say so and name the fields, or it starts echoing field names into the report — reports
  that literally contain "executive_summary:" are the classic symptom.


In [ ]:
# ==============================================================================
# ADDED · EX-2
# the pydantic schema that types the analyst -> writer handoff
# ----------------------------------------------------------------------------
# Not part of the original lab. Builds its own Task objects; never mutates the
# original research_task / analysis_task / writing_task.
# ==============================================================================

from typing import List, Literal

from pydantic import BaseModel, Field


class Implication(BaseModel):
    """One ranked implication. `rank` is what makes the ordering checkable in code."""

    rank: int = Field(description="1 = highest impact. Contiguous, no ties, no gaps.")
    claim: str = Field(description="The implication, one sentence.")
    reasoning: str = Field(description="Why it follows from the research. One or two sentences.")
    impact: Literal["high", "medium", "low"] = Field(description="Magnitude if it holds.")
    # A free-text field would let the model write "probably true"; a Literal forces a commitment
    # to one of three values a downstream consumer can branch on.
    confidence: Literal["high", "medium", "low"] = Field(
        description="How well the audited research supports this. LOW if it rests on a flagged claim."
    )


class Recommendation(BaseModel):
    action: str = Field(description="A concrete action, phrased as an imperative.")
    rationale: str = Field(description="The implication it answers and why it is worth doing.")


class AnalysisOutput(BaseModel):
    """The typed contract between analyst and writer.

    `dropped_claims` exists so the fact-checker's audit trail survives the schema. Without a
    field to land in, a constrained output silently deletes the very safety feature Exercise 1
    added -- the schema decides what can be said, so anything unnamed here cannot be passed on.
    """

    executive_summary: str = Field(description="Exactly two sentences.")
    implications: List[Implication] = Field(description="Three, ranked, most impactful first.")
    recommendations: List[Recommendation] = Field(description="Two.")
    dropped_claims: List[str] = Field(
        default_factory=list,
        description="Claims from the fact-checker's 'do not repeat downstream' list that you "
                    "deliberately left out, each with a few words on why.",
    )


print("Schema defined. Fields the writer will receive:")
for name, field in AnalysisOutput.model_fields.items():
    print(f"  {name:<20} {field.annotation}")

In [ ]:
# ==============================================================================
# ADDED · EX-2
# the typed crew: output_pydantic on the analysis task
# ----------------------------------------------------------------------------
# Not part of the original lab. Builds its own Task objects; never mutates the
# original research_task / analysis_task / writing_task.
# ==============================================================================

# Exercise 2 builds on Exercise 1's four-agent shape, so the typed analysis also carries the
# audit forward. Fresh Task objects again -- Exercise 1's outputs stay intact for comparison.

ex2_research_task = make_research_task()

ex2_factcheck_task = Task(
    description=ex1_factcheck_task.description,   # unchanged: the auditor is not the variable here
    expected_output=ex1_factcheck_task.expected_output,
    agent=fact_checker,
    context=[ex2_research_task],
)

ex2_analysis_task = Task(
    description="""# Context
Two documents are available to you as context: the researcher's brief, and the fact-checker's
audit of that brief.

# Task
Analyze the research and produce ranked implications and recommendations.

# Constraints
- The audit OVERRIDES the brief. Where they disagree, the audit wins.
- Rank implications by impact: rank 1 is the most impactful. Ranks must be 1, 2, 3 -- no ties,
  no gaps.
- Set `confidence` to "low" for any implication that rests on a claim the audit flagged.
- Put every claim you deliberately left out into `dropped_claims`, with a few words on why.
  Do not silently discard an audit finding.

# Format
Return ONLY a JSON object matching the required schema. No markdown fence, no prose around it.""",
    expected_output=(
        "A JSON object with keys: executive_summary (string, exactly two sentences); "
        "implications (array of exactly 3 objects, each with rank:int, claim:string, "
        "reasoning:string, impact:'high'|'medium'|'low', confidence:'high'|'medium'|'low'); "
        "recommendations (array of exactly 2 objects, each with action:string, rationale:string); "
        "dropped_claims (array of strings)."
    ),
    agent=analyst,
    context=[ex2_research_task, ex2_factcheck_task],
    output_pydantic=AnalysisOutput,   # <-- the whole exercise
)

ex2_writing_task = Task(
    description="""# Context
Two documents are available to you as context: the fact-checker's audit (markdown), and the
analysis -- which arrives as a JSON OBJECT, not prose. Its fields are:

- `executive_summary` -- two sentences framing the whole picture
- `implications` -- three objects, each with `rank` (1 = most important), `claim`, `reasoning`,
  `impact`, and `confidence`
- `recommendations` -- two objects, each with `action` and `rationale`
- `dropped_claims` -- claims the analyst deliberately excluded; these are OFF LIMITS to you

# Task
Write a polished, publication-ready short report for a general audience.

# Constraints
- 200-300 words; clear and engaging; no jargon dumps.
- Lead with the rank-1 implication. Rank order is the analyst's judgment of importance -- honour it.
- Treat a `confidence` of "low" as a hedge you must carry into the prose, not drop.
- Say nothing that appears in `dropped_claims`.
- Never print a field name. The reader must never see "executive_summary" or "rank" -- you are
  writing prose from structured data, not transcribing it.
- If you use a figure, keep it exact and name its source in the same sentence. Never swap a
  figure for a vague quantifier ("nearly a third", "most", "the majority").

# Format
A title, then 2-3 short paragraphs.""",
    expected_output="A short titled report of 200-300 words, with no field names visible.",
    agent=writer,
    context=[ex2_factcheck_task, ex2_analysis_task],
)

ex2_crew = Crew(
    agents=[researcher, fact_checker, analyst, writer],
    tasks=[ex2_research_task, ex2_factcheck_task, ex2_analysis_task, ex2_writing_task],
    process=Process.sequential,
    verbose=False,
)

ex2_ok = await run_crew(ex2_crew, "the typed-handoff crew")
if ex2_ok:
    print("=" * 70)
    print("=== TYPED ANALYSIS -- accessed as an OBJECT, not parsed from text ===")
    print("=" * 70)
    analysis = ex2_analysis_task.output.pydantic
    if analysis is None:
        # Version-dependent: some CrewAI/LiteLLM combinations fill .json_dict but not .pydantic.
        print("output.pydantic is None -- falling back to json_dict. Check your CrewAI version.")
        analysis = AnalysisOutput(**ex2_analysis_task.output.json_dict)

    print(f"type: {type(analysis).__name__}\n")
    print(f"summary: {analysis.executive_summary}\n")
    for imp in analysis.implications:
        print(f"  #{imp.rank} [impact={imp.impact}, confidence={imp.confidence}] {imp.claim}")
        print(f"      why: {imp.reasoning}")
    print()
    for rec in analysis.recommendations:
        print(f"  -> {rec.action}\n     ({rec.rationale})")
    print(f"\ndropped_claims ({len(analysis.dropped_claims)}):")
    for claim in analysis.dropped_claims or ["(none)"]:
        print(f"  - {claim}")

    print("\n" + "=" * 70)
    print("=== FINAL REPORT (writer, from typed input) ===")
    print("=" * 70)
    print(ex2_writing_task.output.raw)

    save_output("typed-handoff/analysis.json", ex2_analysis_task.output.raw)
    save_output("typed-handoff/factcheck_audit.md", ex2_factcheck_task.output.raw)
    save_output("typed-handoff/final_report.md", ex2_writing_task.output.raw)
    print(f"\nSaved to {OUT_ROOT}/typed-handoff/")

---

### `[ADDED · EX-2]` Check: what the schema buys you that prose does not
Every assertion below is **impossible to write against the prose version** of the analysis. That
is the entire argument for `output_pydantic`, stated as code rather than as a claim.


In [ ]:
# ==============================================================================
# ADDED · EX-2
# assertions that are impossible against the prose version
# ----------------------------------------------------------------------------
# Not part of the original lab. Builds its own Task objects; never mutates the
# original research_task / analysis_task / writing_task.
# ==============================================================================

if not ex2_ok or task_text(ex2_writing_task) is None:
    print("Run the Exercise 2 crew cell first.")
else:
    analysis = ex2_analysis_task.output.pydantic or AnalysisOutput(**ex2_analysis_task.output.json_dict)
    report = ex2_writing_task.output.raw

    checks = {}

    # 1. Ranks are contiguous 1..N. Unverifiable in prose; a one-liner here.
    ranks = [i.rank for i in analysis.implications]
    checks["ranks are a contiguous 1..N with no ties or gaps"] = sorted(ranks) == list(range(1, len(ranks) + 1))

    # 2. Cardinality the prompt asked for. In prose this is a suggestion.
    checks["exactly 3 implications"] = len(analysis.implications) == 3
    checks["exactly 2 recommendations"] = len(analysis.recommendations) == 2

    # 3. Enum fields really are in the enum -- pydantic would have rejected them otherwise,
    #    which is the point: the failure happened at the boundary, not downstream.
    checks["impact/confidence are valid enum values"] = all(
        i.impact in ("high", "medium", "low") and i.confidence in ("high", "medium", "low")
        for i in analysis.implications
    )

    # 4. Two sentences means two sentences.
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", analysis.executive_summary.strip()) if s]
    checks["executive_summary is exactly 2 sentences"] = len(sentences) == 2

    # 5. No field name leaked into the prose report.
    leaked = [f for f in AnalysisOutput.model_fields if f in report] + \
             [f for f in ("rank", "confidence", "rationale") if re.search(rf"\b{f}\s*[:=]", report)]
    checks["no schema field names leaked into the report"] = not leaked

    # 6. Dropped claims stayed dropped. The audit trail survived BOTH the schema and the writer.
    still_present = []
    for claim in analysis.dropped_claims:
        # Compare on the claim's distinctive words, since the writer would paraphrase not quote.
        key_words = [w for w in re.findall(r"[a-zA-Z]{6,}", claim)][:3]
        if key_words and all(w.lower() in report.lower() for w in key_words):
            still_present.append(claim)
    checks["claims the analyst dropped do not reappear in the report"] = not still_present

    # 7. The report still obeys the word budget.
    words = len(report.split())
    checks[f"report is 200-300 words (got {words})"] = 200 <= words <= 300

    for name, passed in checks.items():
        print(f"  [{'PASS' if passed else 'FAIL'}] {name}")
    if leaked:
        print(f"\n  leaked field names: {sorted(set(leaked))}")
    if still_present:
        print(f"\n  dropped claims that came back: {still_present}")

    print()
    print("Now try the same assertions against outputs/before-memory/analysis.md. You cannot --")
    print("there is no rank to compare, no enum to validate, and no list of dropped claims. You")
    print("would be writing a parser and guessing, which is the failure mode the schema removes.")

---

## `[ADDED · EX-3]` Exercise 3 — `Process.hierarchical` (deferred)

Not attempted in this pass, by choice.

Worth recording what it would take, since the design decision is the interesting part: a
hierarchical crew needs `manager_llm` **or** `manager_agent` (never both), the manager must not
appear in `agents=`, and the delegating agents need `allow_delegation=True`.

The trap is subtler. If the tasks keep their explicit `context=[...]` wiring, the manager has
nothing left to decide and a hierarchical run just looks like a sequential run with a large extra
token bill. Showing anything real means building **fresh tasks with no `context=`** and letting
the manager's delegation *be* the wiring — then comparing `crew.usage_metrics` against the
sequential baseline to price what that autonomy costs.


---

## `[ADDED · EX-4]` Exercise 4 — prove memory works

### Why the optional cell above does not prove it

The `memory_crew` demo earlier runs the same crew, on the same topic, with the **same task
descriptions**, twice — and then invites you to notice that Run 2 resembles Run 1. But identical
prompts to a temperature-0.3 model already produce similar output. Every bit of that resemblance
is explained without memory, so the demo cannot distinguish "the crew recalled Run 1" from "the
crew was asked the same question twice". It also overwrites `writing_task.output`, destroying the
baseline objects in the process.

Proving it needs two things the demo lacks:

**1. Run 2 must ask for something only memory can supply.** So Run 2's prompt is *different*: it
tells the crew it has studied this topic before and asks it to open with a section listing the
specific systems and figures it established last time, then extend rather than repeat them. If
memory works, that section contains Run 1's actual specifics. If it doesn't, the crew either
admits it has nothing or invents something — and those two failure modes are distinguishable,
which matters.

**2. A control.** The same two runs, the same prompts, `memory=False`. Without it there is no way
to know whether an overlap between Run 2 and Run 1 came from memory or from both runs simply
being about the same topic — which on a shared topic is most of it. The control establishes that
floor, and only the **difference** between the arms is evidence.

### Two details that decide whether this works at all

- **`CREWAI_STORAGE_DIR` is set and wiped first.** By default CrewAI writes memory to a hidden
  platform app-data directory. Pointing it somewhere visible makes the store inspectable *and*
  wipeable — and the wipe matters, because the optional cell above may already have seeded it
  with runs of this exact topic. Leftover state would leak straight into the control's arm of
  the comparison.
- **Each arm mutates `crew.tasks` between runs instead of building a second crew.** A `Crew`
  builds its memory objects at construction, and in some CrewAI versions short-term memory is
  scoped to that instance. Reusing the instance gives memory its best possible shot; anything it
  fails to recall this way, it would not have recalled either. The control performs the identical
  mutation, so the two arms differ in exactly one flag.

> **Runtime:** four full three-agent runs, ~12 LLM calls plus embedding calls. Give it a few minutes.


In [ ]:
# ==============================================================================
# ADDED · EX-4
# two arms: memory=False control, then memory=True
# ----------------------------------------------------------------------------
# Not part of the original lab. Builds its own Task objects; never mutates the
# original research_task / analysis_task / writing_task.
# ==============================================================================

import shutil

# Must be set BEFORE any memory-backed Crew is constructed -- CrewAI resolves the storage path
# when it builds its memory objects. If you have already run a memory crew in this session,
# restart the runtime to be certain this takes effect.
STORAGE_DIR = os.path.abspath("crewai_lab03b_storage")
os.environ["CREWAI_STORAGE_DIR"] = STORAGE_DIR

MEM_TOPIC = TOPIC   # same topic as the baseline, so the control's overlap floor is realistic


def make_run1_tasks():
    """Run 1: an ordinary run. Identical wording in both arms."""
    research = make_research_task(MEM_TOPIC)
    analysis = Task(
        description="""# Context
The researcher's brief is available to you as context.

# Task
Analyze the research and draw the most important implications and recommendations.

# Constraints
- Rank implications by impact; back each claim with reasoning.

# Format
Executive Summary (2 sentences), Key Implications (3 bullets), Recommendations (2 bullets).""",
        expected_output="A ranked analysis with a summary, implications, and recommendations.",
        agent=analyst,
        context=[research],
    )
    writing = Task(
        description="""# Context
The research brief AND the analysis are available to you as context.

# Task
Write a polished, publication-ready short report that combines them for a general audience.

# Constraints
- 200-300 words; clear and engaging; no jargon dumps.

# Format
A title, then 2-3 short paragraphs.""",
        expected_output="A short titled report of 200-300 words.",
        agent=writer,
        context=[research, analysis],
    )
    return [research, analysis, writing]


def make_run2_tasks():
    """Run 2: asks for something ONLY memory can supply.

    The '## What I established last time' section is the measurement surface. With memory it
    should name Run 1's actual systems and figures; without it, the crew must either say it has
    nothing (honest) or invent specifics (confabulation). Both outcomes are informative, and the
    entity-overlap check below tells them apart.
    """
    research = Task(
        description=f"""# Context
You have researched this topic before, in an earlier session. Build on what you found last time
rather than starting over.

# Task
Extend your earlier research on the topic in the Input section.

# Constraints
- Begin with a section headed exactly "## What I established last time" listing the SPECIFIC
  systems, benchmarks, examples and figures you covered previously -- by name and by number.
- If you genuinely cannot recall anything specific from an earlier session, say exactly
  "NO PRIOR CONTEXT RECALLED" under that heading. Do NOT guess or reconstruct what you probably
  said -- an invented recollection is worse than an admitted blank.
- Then cover what you did NOT cover last time. Do not repeat the earlier ground.

# Format
The recall section, then short sections with clear headers.

# Input (topic)
{MEM_TOPIC}""",
        expected_output="A recall section, then new research extending the earlier brief.",
        agent=researcher,
    )
    analysis = Task(
        description="""# Context
The researcher's extended brief is available to you as context.

# Task
Analyze what is NEW relative to the earlier session and draw the implications.

# Constraints
- Rank implications by impact; back each claim with reasoning.
- Note explicitly which implications are new since last time.

# Format
Executive Summary (2 sentences), Key Implications (3 bullets), Recommendations (2 bullets).""",
        expected_output="A ranked analysis of what is new.",
        agent=analyst,
        context=[research],
    )
    writing = Task(
        description="""# Context
The extended brief AND the analysis are available to you as context.

# Task
Write a short follow-up report that builds on the earlier one.

# Constraints
- 200-300 words.
- Reference the earlier findings by name where you build on them.

# Format
A title, then 2-3 short paragraphs.""",
        expected_output="A short titled follow-up report of 200-300 words.",
        agent=writer,
        context=[research, analysis],
    )
    return [research, analysis, writing]


EMBEDDER = {
    "provider": "google-generativeai",
    "config": {"api_key": api_key, "model_name": "gemini-embedding-001"},
}

results = {}   # arm -> {"run1": <brief>, "run2": <brief>, "run2_report": <report>}


async def run_arm(arm: str, use_memory: bool):
    """Run 1 then Run 2 through ONE crew instance, so memory keeps its instance-scoped state."""
    print(f"\n{'=' * 70}\nARM: {arm}  (memory={use_memory})\n{'=' * 70}")
    run1 = make_run1_tasks()
    kwargs = dict(agents=[researcher, analyst, writer], process=Process.sequential, verbose=False)
    if use_memory:
        kwargs.update(memory=True, embedder=EMBEDDER)
    crew = Crew(tasks=run1, **kwargs)

    print("  Run 1 ...")
    if not await run_crew(crew, f"{arm} run 1"):
        return
    run1_brief = run1[0].output.raw

    # Same crew instance, new tasks -- this is what gives memory its best shot at recall.
    run2 = make_run2_tasks()
    crew.tasks = run2
    print("  Run 2 (asked to build on last time) ...")
    if not await run_crew(crew, f"{arm} run 2"):
        return

    results[arm] = {
        "run1": run1_brief,
        "run2": run2[0].output.raw,
        "run2_report": run2[2].output.raw,
    }
    for name, text in [("run1_brief", run1_brief), ("run2_brief", run2[0].output.raw),
                       ("run2_report", run2[2].output.raw)]:
        save_output(f"exercise4-{arm}/{name}.md", text)
    print(f"  done -- saved to {OUT_ROOT}/exercise4-{arm}/")


# CONTROL FIRST, while no memory store exists at all. Ordering it this way means the control
# cannot be contaminated even if the wipe below were to fail.
await run_arm("control-no-memory", use_memory=False)

# Now wipe and run the memory arm.
if os.path.isdir(STORAGE_DIR):
    shutil.rmtree(STORAGE_DIR)
    print(f"\nWiped {STORAGE_DIR}")
await run_arm("with-memory", use_memory=True)

print("\nArms completed:", sorted(results))

---

### `[ADDED · EX-4]` The measurement
Three signals, weakest to strongest:

1. **Backreference phrases** ("last time", "previously") — nearly worthless on its own. The Run 2
   prompt *tells* the crew it has been here before, so both arms will say "previously" whether or
   not anything was recalled. Reported only to make that point concrete.
2. **Entity recall** — the fraction of Run 1's distinctive entities (named systems, benchmarks,
   figures) that reappear in Run 2. This has a floor above zero in the control, because both runs
   are about the same topic and any competent researcher names SWE-bench. That floor is exactly
   why the control exists; the number that means something is **memory recall minus control recall**.
3. **The explicit blank** — whether the crew wrote `NO PRIOR CONTEXT RECALLED`. The control
   *should*. If it instead lists confident specifics, it is confabulating, and the entity overlap
   tells us whether those specifics were real.

A note on reading the result: if the delta is small or zero, that is a finding, not a broken cell.
CrewAI's short-term memory is scoped per-`kickoff` in some versions, and long-term memory stores
task-level evaluations rather than the research content itself — so a null result is entirely
plausible and is the honest input to Exercise 5. What would be dishonest is rewriting the Run 2
prompt until it *looks* like recall.


In [ ]:
# ==============================================================================
# ADDED · EX-4
# measure recall: entity overlap, memory arm vs control
# ----------------------------------------------------------------------------
# Not part of the original lab. Builds its own Task objects; never mutates the
# original research_task / analysis_task / writing_task.
# ==============================================================================

STOPWORDS = {
    "the", "and", "that", "this", "with", "from", "have", "these", "those", "their", "there",
    "which", "while", "where", "when", "will", "would", "could", "should", "been", "being",
    "than", "then", "them", "they", "what", "such", "into", "more", "most", "some", "also",
    "about", "because", "however", "furthermore", "moreover", "therefore", "software",
    "development", "developer", "developers", "agent", "agents", "workflow", "workflows",
    "code", "coding", "impact", "context", "session", "research", "brief", "report", "analysis",
    "last", "time", "previously", "earlier", "established",
}

# An "entity" is the kind of specific a paraphrase does NOT preserve: a named system
# (CamelCase or hyphenated-capitalised), or a figure. Lowercase prose words are excluded
# deliberately -- two documents on one topic share those regardless of memory.
ENTITY_RE = re.compile(r"\b[A-Z][A-Za-z0-9]*(?:[-\u2011][A-Za-z0-9]+)*\b")


def entities(text: str) -> set:
    """Distinctive named things plus figures, normalized for comparison."""
    found = set()
    for m in ENTITY_RE.finditer(text):
        tok = m.group()
        if len(tok) < 4 or tok.lower() in STOPWORDS:
            continue
        # Skip a capitalised common word that only got its capital from starting a sentence:
        # require either an internal capital, a hyphen, or a digit -- the marks of a real name.
        if not (re.search(r"[A-Z0-9]", tok[1:]) or "-" in tok or "\u2011" in tok):
            continue
        found.add(tok.lower())
    found |= figures(text)
    return found


BACKREF_RE = re.compile(
    r"\b(?:last time|previously|earlier session|prior session|as (?:I|we) (?:found|noted|established)|"
    r"in my earlier|building on|previous(?:ly)? (?:brief|research|report)|before)\b", re.I
)


def recall_section(text: str) -> str:
    """Just the '## What I established last time' block, if the model produced one."""
    m = re.search(r"##\s*What I established last time(.*?)(?=\n##\s|\Z)", text, re.S | re.I)
    return m.group(1).strip() if m else ""


if len(results) < 2:
    print("Both arms have not completed. Re-run the cell above (a 503 may have cut one short).")
else:
    print("=" * 78)
    print(f"{'':<34}{'control (memory=False)':<24}{'with-memory':<20}")
    print("=" * 78)

    rows = {}
    for arm in ("control-no-memory", "with-memory"):
        r = results[arm]
        e1, e2 = entities(r["run1"]), entities(r["run2"])
        section = recall_section(r["run2"])
        sec_ents = entities(section) if section else set()
        rows[arm] = {
            "run1 entities": len(e1),
            "shared run1->run2": len(e1 & e2),
            "entity recall": (len(e1 & e2) / len(e1)) if e1 else 0.0,
            "recall-section entities": len(sec_ents),
            "of those, real (in run1)": len(sec_ents & e1),
            "confabulated in section": len(sec_ents - e1),
            "backref phrases": len(BACKREF_RE.findall(r["run2"])),
            "declared blank": "NO PRIOR CONTEXT RECALLED" in r["run2"].upper(),
        }

    for metric in rows["control-no-memory"]:
        c, m_ = rows["control-no-memory"][metric], rows["with-memory"][metric]
        fmt = (lambda v: f"{v:.0%}") if metric == "entity recall" else (lambda v: str(v))
        print(f"  {metric:<32}{fmt(c):<24}{fmt(m_):<20}")

    delta = rows["with-memory"]["entity recall"] - rows["control-no-memory"]["entity recall"]
    print("=" * 78)
    print(f"  ENTITY-RECALL DELTA (memory - control): {delta:+.0%}")
    print("  ^ this, not the raw recall number, is the evidence.")
    print()

    if delta >= 0.15:
        print("VERDICT: memory shows a real effect. Run 2 in the memory arm carried forward")
        print("specifics that the control -- same prompts, same topic -- did not.")
    elif delta > 0.05:
        print("VERDICT: a weak positive. Directionally right but within what one sample's variance")
        print("could produce. Re-run both arms before claiming it; n=1 is not evidence.")
    else:
        print("VERDICT: NO measurable memory effect on this run.")
        print("This is a legitimate result and worth reporting as-is. Likely explanations:")
        print("  - CrewAI short-term memory is scoped per-kickoff in this version, so Run 2's")
        print("    crew never queried Run 1's store despite sharing the instance.")
        print("  - Long-term memory stores task-level quality evaluations, not the research")
        print("    content -- so there is nothing there for a researcher to recall.")
        print("  - Retrieval is by embedding similarity with a relevance threshold; the Run 2")
        print("    prompt may simply not have retrieved anything above it.")
        print("Do NOT tune the Run 2 prompt until it looks like recall -- that measures the")
        print("prompt, not the memory.")

    # Confabulation is the finding people miss: the control was told it had a past and often
    # invents one rather than taking the blank the prompt explicitly offered.
    ctrl = rows["control-no-memory"]
    if not ctrl["declared blank"] and ctrl["confabulated in section"] > 0:
        print()
        print(f"NOTE: the control did NOT take the offered blank and instead named "
              f"{ctrl['confabulated in section']} specifics absent from its own Run 1. Told it had a")
        print("history, it manufactured one. Anything relying on a model's self-report of what it")
        print("remembers is measuring compliance with the prompt, not recall.")

    print()
    print("=" * 78)
    print("WHAT IS ACTUALLY ON DISK")
    print("=" * 78)
    if os.path.isdir(STORAGE_DIR):
        for root, _dirs, files in os.walk(STORAGE_DIR):
            for f in sorted(files):
                p = os.path.join(root, f)
                print(f"  {os.path.relpath(p, STORAGE_DIR):<58}{os.path.getsize(p):>10,} bytes")
    else:
        print(f"  {STORAGE_DIR} does not exist -- nothing was persisted.")
        print("  If the memory arm ran, CREWAI_STORAGE_DIR was probably resolved before this")
        print("  cell set it. Restart the runtime and re-run from the setup cell.")

---

## `[ADDED · EX-5]` Exercise 5 — framework memory vs. hand-rolled memory

I built memory by hand in Lab 03A and got it from the framework here. They are not the same
mechanism wearing different clothes, and the difference is sharper than "one is more convenient".

### What each one actually is

**Lab 03A** — memory is a Markdown file, `/content/reflexion_memory.md`, wrapped by
`ReflexionMemory` with exactly two doors: `remember(lesson)` appends a line, `recall(last_n)`
reads lines back. What gets stored is a *deliberate distillation*: the reflector turns a critique
into one sentence starting "Next attempt:", and that sentence is the memory. Retrieval is
last-N — deterministic, ordered, and complete.

**Lab 03B** — memory is `memory=True`. CrewAI stands up three stores (short-term, long-term,
entity), embeds content through Gemini, and retrieves by vector similarity. What gets stored, I
did not choose. What comes back, I cannot predict.

### The comparison

| | Lab 03A (by hand) | Lab 03B (CrewAI) |
|---|---|---|
| Lines of code | ~40 for the class + reflector | one kwarg |
| What is stored | one distilled sentence per iteration, by my design | framework's choice across 3 stores |
| Where it lives | a Markdown file I can `cat` | SQLite + Chroma under `CREWAI_STORAGE_DIR` |
| Retrieval | last-N: deterministic, ordered | embedding similarity above a threshold |
| Inspectable | open the file | dump a vector DB and infer |
| Testable | `recall()` is a pure function — assert on it | assert on *whether recall happened*, indirectly |
| Clearing stale memory | `memory.reset()`, a documented method | find and delete a storage directory |
| Failure mode | wrong lesson stored — visible in the file | nothing retrieved — silent, looks like normal output |
| Cost | zero extra calls | embedding calls per store and per retrieval |
| Portability | a Markdown file | CrewAI's schema |

### The thing I did not expect

The framework's memory is **harder to debug than the hand-rolled one, in the specific way that
matters most**: when it silently does nothing, the output still looks fine. Exercise 4 needed a
control arm, an entity-overlap metric, and four crew runs to answer "did it recall anything?" —
and even then the answer is statistical. In 03A that question is `print(memory.recall())`.

That asymmetry generalizes past memory. The same invisibility produced the drift Exercise 1 fixed:
`context=[research_task]` is a beautifully compact way to express a handoff, and it is precisely
because the handoff is one word that nobody looks at what crosses it. A number lost its source
between the researcher and the analyst, then lost its precision between the analyst and the writer,
and the crew reported success. Hand-rolling that chain, I would have been holding the intermediate
string in a variable and would likely have read it.

So the framework did not save me the work — it moved the work from *writing the mechanism* to
*proving the mechanism ran*. On this lab, the second job was bigger.

### Which I would reach for in production

**CrewAI for the orchestration; something explicit for the memory.**

The crew abstraction genuinely earns its place. Sequential handoff, per-task context declaration,
`output_pydantic` at the boundary, four agents wired in a dozen lines — I would not hand-roll that,
and Exercise 2 shows the payoff: a typed boundary turns "rank by impact" from a hope into an
assertion. That is the framework doing real work.

`memory=True` I would not ship as-is, for one reason: **I cannot answer "why did the agent say
that?"** In production that question arrives attached to an incident. A store I did not design,
holding content I did not choose, retrieved by similarity I cannot replay, is an input to my
system that I cannot reconstruct after the fact. The lab's own warning — memory is "powerful and
sticky", and can "carry stale or wrong facts forward" — describes a risk you can only manage if
you can *see* what is in there.

What I would actually build is 03A's shape at 03B's scale: an explicit store whose write path I
control, whose contents are readable, and whose retrieval is deterministic enough to reproduce.
Not a Markdown file — a real datastore with retrieval I chose. But the same property, which is the
one 03A had and `memory=True` gave up: **I can read the memory and say what the agent knew.**

The honest summary: 03A was clearer because I built it, and I would keep that clarity for state
that outlives a run. 03B was faster because I did not, and that trade is fine for the
orchestration, where a mistake shows up in the output. For memory, a mistake shows up as
confidence — which is why I would keep that piece in my own hands.
